# 10c — Modules, Gene Enrichment & Motif Enrichment

**Goal:** Extract K-means modules from the cross-cell-type signed-p-value matrix,
run gene enrichment (GO, GREAT), and parallel monaLisa motif enrichment.
Also run motif enrichment on ultra-robust evolutionary branch peaks.

**Requires:** Intermediate files from notebooks `10a` and `10b`.

**Outputs:**
- `differential_results/cross_celltype_modules/` — module CSVs + BED files
- `differential_results/enrichment/` — GO dotplots + GREAT CSVs per module/cell-type
- `differential_results/motifs/` — monaLisa SE objects + heatmap PDFs
- `plots/Cross_CellType_ComplexHeatmap_Modules.pdf`
- `plots/Motif_vs_CellType_Heatmap.pdf`
- `plots/Motif_vs_Module_Heatmap.pdf`
- `plots/Motif_vs_Branch_Heatmap.pdf`

In [1]:
# =============================================================================
# Cell 1: Load Packages & Source Utilities
# =============================================================================
suppressPackageStartupMessages({
  library(DESeq2)
  library(ggplot2)
  library(dplyr)
  library(readr)
  library(ComplexHeatmap)
  library(circlize)
})

source("../src/deseq2_utils.R")
message("Packages loaded & utilities sourced.")

Packages loaded & utilities sourced.



In [2]:
# =============================================================================
# Cell 2: Configuration & Load Intermediate Data
# =============================================================================
BASE      <- "/cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks"
OUT_DIR   <- file.path(BASE, "cross_species_consensus_v3/13_deseq2_R_pseudobulk")
SPECIES   <- c("Human", "Bonobo", "Chimpanzee", "Gorilla", "Macaque", "Marmoset")

# Signed-pvalue matrix from 10b (shared peaks only)
plot_mat_shared <- readRDS(file.path(OUT_DIR, "differential_results",
                                     "Signed_Pval_Matrix_shared.rds"))

# DESeq2 results: shared peaks
res_list_shared <- readRDS(file.path(OUT_DIR, "differential_results",
                                     "DESeq2_res_list_shared_all.rds"))

# DESeq2 results: evolutionary branches + ultra-robust
branch_res    <- readRDS(file.path(OUT_DIR, "differential_results",
                                   "evolutionary_branches", "branch_res_list.rds"))
robust_peaks  <- readRDS(file.path(OUT_DIR, "differential_results",
                                   "ultra_robust_branches", "ultra_robust_peaks_list.rds"))

# Load master annotation
anno_file   <- file.path(BASE, "cross_species_consensus_v3/07_master_annotation/master_annotation_final.tsv")
master_anno <- read_tsv(anno_file, show_col_types = FALSE)
options(scipen = 999)

message("Shared signed-p matrix:   ", nrow(plot_mat_shared), " regions x ",
        ncol(plot_mat_shared), " cell types")
message("Shared DESeq2 results:    ", length(res_list_shared), " cell types")
message("Branch results:           ", length(branch_res), " cell types")
message("Ultra-robust peak sets:   ", length(robust_peaks), " cell types")

Shared signed-p matrix:   27131 regions x 13 cell types

Shared DESeq2 results:    13 cell types

Branch results:           13 cell types

Ultra-robust peak sets:   13 cell types



## K-Means Module Extraction

In [ ]:
# =============================================================================
# Cell 3: Extract K-Means Modules (from Shared-Peak Matrix)
# =============================================================================
NUM_MODULES <- 40
MIN_REGIONS <- 5

mod_result <- extract_kmeans_modules(
  plot_mat_shared,
  num_modules = NUM_MODULES,
  min_regions = MIN_REGIONS,
  out_dir     = OUT_DIR,
  seed        = 42
)

km_res       <- mod_result$km_res
master_df    <- mod_result$master_df
keep_modules <- mod_result$keep_modules

message("\nModules extracted: ", length(keep_modules),
        " modules, ", nrow(master_df), " total regions")

In [ ]:
# =============================================================================
# Cell 4: Generate BED Files per Module
# =============================================================================
cluster_dir <- file.path(OUT_DIR, "differential_results", "cross_celltype_modules")

for (mod_num in keep_modules) {
  mod_peaks <- master_df$region_id[master_df$module == paste0("Module_", mod_num)]

  write_peaks_bed(mod_peaks, "Human", master_anno,
                  file.path(cluster_dir, paste0("Module_", mod_num, "_regions.bed")))
}

message("Module BED files generated.")

## Gene Enrichment (GO & GREAT)

In [ ]:
# =============================================================================
# Cell 5: GO Enrichment per Module
# =============================================================================
suppressPackageStartupMessages({
  library(clusterProfiler)
  library(org.Hs.eg.db)
})

for (mod_num in keep_modules) {
  mod_peaks <- master_df$region_id[master_df$module == paste0("Module_", mod_num)]

  # Map peaks to human genes
  gene_info <- get_peak_info(mod_peaks, "Human", "gene", master_anno)
  mod_genes <- unique(gene_info$gene)
  mod_genes <- mod_genes[mod_genes != "." & !is.na(mod_genes)]

  run_go_enrichment(mod_genes,
                    label   = paste0("Module_", mod_num),
                    out_dir = OUT_DIR)
}

message("\nGO enrichment complete for all modules.")

In [ ]:
# =============================================================================
# Cell 6: GO Enrichment per Cell Type (Human UP peaks)
# =============================================================================
for (ct in names(res_list_shared)) {
  if (!"Human" %in% names(res_list_shared[[ct]])) next

  res_df <- as.data.frame(res_list_shared[[ct]][["Human"]])
  up_peaks <- rownames(res_df)[!is.na(res_df$padj) &
                                res_df$padj < 0.05 &
                                res_df$log2FoldChange > 1]

  gene_info <- get_peak_info(up_peaks, "Human", "gene", master_anno)
  genes <- unique(gene_info$gene)
  genes <- genes[genes != "." & !is.na(genes)]

  run_go_enrichment(genes,
                    label   = paste0(ct, "_Human_UP"),
                    out_dir = OUT_DIR)
}

message("GO enrichment complete for cell-type Human-UP peaks.")

In [ ]:
# =============================================================================
# Cell 7: GREAT Analysis for Representative Modules
# =============================================================================
suppressPackageStartupMessages(library(rGREAT))

# Run GREAT for each module
for (mod_num in keep_modules) {
  mod_peaks <- master_df$region_id[master_df$module == paste0("Module_", mod_num)]

  run_great_analysis(
    peak_ids = mod_peaks,
    species  = "Human",
    anno_df  = master_anno,
    label    = paste0("Module_", mod_num),
    out_dir  = OUT_DIR,
    genome   = "hg38"
  )
}

message("GREAT analysis complete.")

## Motif Enrichment (Parallel monaLisa)

In [ ]:
# =============================================================================
# Cell 8: Motif Enrichment — Cell Types (Human UP, Parallel)
# =============================================================================
source("../src/deseq2_utils.R")   # picks up SnowParam fix

N_CORES <- 10L  # Adjust based on your cluster allocation

motif_dir <- file.path(OUT_DIR, "differential_results", "motifs")
dir.create(motif_dir, showWarnings = FALSE, recursive = TRUE)

# Gather Human-UP peaks per cell type (>= 20 peaks required)
ct_peaks <- list()
for (ct in names(res_list_shared)) {
  if (!"Human" %in% names(res_list_shared[[ct]])) next
  res_df <- as.data.frame(res_list_shared[[ct]][["Human"]])
  up_peaks <- rownames(res_df)[!is.na(res_df$padj) &
                                res_df$padj < 0.05 &
                                res_df$log2FoldChange > 1]
  if (length(up_peaks) >= 20) ct_peaks[[ct]] <- up_peaks
}

message("Running parallel motif enrichment for ", length(ct_peaks), " cell types...")

se_ct <- run_parallel_motif_enrichment(
  peak_list_named = ct_peaks,
  anno_df         = master_anno,
  species         = "Human",
  n_cores         = N_CORES,
  out_rds         = file.path(motif_dir, "CellType_SE_Object.rds")
)

plot_motif_heatmap(se_ct,
                   title    = "Top Motifs per Human-UP Cell Type",
                   out_file = file.path(OUT_DIR, "plots",
                                        "Motif_vs_CellType_Heatmap.pdf"))

message("Cell-type motif enrichment complete.")

In [ ]:
# =============================================================================
# Cell 9: Motif Enrichment — K-Means Modules (Parallel)
# =============================================================================
# Gather module peak lists (>= 20 peaks required)
mod_peaks_list <- split(master_df$region_id, master_df$module)
mod_peaks_list <- mod_peaks_list[sapply(mod_peaks_list, length) >= 20]

message("Running parallel motif enrichment for ", length(mod_peaks_list), " modules...")

se_mod <- run_parallel_motif_enrichment(
  peak_list_named = mod_peaks_list,
  anno_df         = master_anno,
  species         = "Human",
  n_cores         = N_CORES,
  out_rds         = file.path(motif_dir, "Module_SE_Object.rds")
)

# Plot
plot_motif_heatmap(se_mod,
                   title    = "Top Motifs per K-Means Module",
                   out_file = file.path(OUT_DIR, "plots",
                                        "Motif_vs_Module_Heatmap.pdf"))

message("Module motif enrichment complete.")

In [3]:
# =============================================================================
# Cell 10: Motif Enrichment — Ultra-Robust Branch Peaks (Parallel)
# =============================================================================
# Re-source in case kernel was restarted from here
if (!exists("motif_dir")) motif_dir <- file.path(OUT_DIR, "differential_results", "motifs")
if (!exists("N_CORES"))   N_CORES   <- 10L
source("../src/deseq2_utils.R")

# Pool ultra-robust peaks per branch x cell-type combination
branch_peak_pool <- list()
for (ct in names(robust_peaks)) {
  for (node in names(robust_peaks[[ct]])) {
    ur <- robust_peaks[[ct]][[node]]
    if (length(ur) == 0) next
    label <- paste0(node, "__", ct)
    branch_peak_pool[[label]] <- ur
  }
}

# Only keep groups with >= 20 peaks for motif enrichment
branch_peak_pool <- branch_peak_pool[sapply(branch_peak_pool, length) >= 20]
message("Branch x cell-type groups for motif enrichment: ", length(branch_peak_pool))

if (length(branch_peak_pool) > 0) {
  message("Running parallel motif enrichment for ", length(branch_peak_pool),
          " branch x cell-type combinations...")

  se_branch <- run_parallel_motif_enrichment(
    peak_list_named = branch_peak_pool,
    anno_df         = master_anno,
    species         = "Human",
    n_cores         = N_CORES,
    out_rds         = file.path(motif_dir, "Branch_UltraRobust_SE_Object.rds")
  )

  plot_motif_heatmap(se_branch,
                     title    = "Top Motifs per Ultra-Robust Branch (by Cell Type)",
                     out_file = file.path(OUT_DIR, "plots",
                                          "Motif_vs_Branch_Heatmap.pdf"))
  message("Ultra-robust branch motif enrichment complete.")
} else {
  message("No branch x cell-type combos with >= 20 ultra-robust peaks; skipping.")
}

Branch x cell-type groups for motif enrichment: 215

Running parallel motif enrichment for 215 branch x cell-type combinations...

Total groups: 215 — processing in batches of 25

Peak memory per batch: ~37.5 MB sequences

Parallel backend: SnowParam (4 workers)

  Batch 1/9: 25 groups, 62904 sequences

Warning message:
“<anonymous>: ... may be used in an incorrect context: 
     findMotifHits(query = pwmL, subject = df$seqs[!df$isForeground], 
         min.score = min.score, method = matchMethod, BPPARAM = BPPARAM, 
         ...)
”
  Batch 2/9: 25 groups, 77222 sequences

Warning message:
“<anonymous>: ... may be used in an incorrect context: 
     findMotifHits(query = pwmL, subject = df$seqs[!df$isForeground], 
         min.score = min.score, method = matchMethod, BPPARAM = BPPARAM, 
         ...)
”
  Batch 3/9: 25 groups, 129702 sequences

Warning message:
“<anonymous>: ... may be used in an incorrect context: 
     findMotifHits(query = pwmL, subject = df$seqs[!df$isForeground], 


## Native monaLisa Plots with Sequence Logos

In [4]:
# =============================================================================
# Cell 11: Native monaLisa Heatmaps with Sequence Logos
# =============================================================================
suppressPackageStartupMessages({
  library(monaLisa)
  library(SummarizedExperiment)
})

plot_monalisa_native <- function(se_obj, filename, top_n = 30) {
  enr_mat <- assay(se_obj, "negLog10Padj")
  enr_mat[is.na(enr_mat)] <- 0

  max_enr       <- apply(enr_mat, 1, max)
  top_motif_idx <- order(max_enr, decreasing = TRUE)[1:min(top_n, nrow(enr_mat))]
  se_sub        <- se_obj[top_motif_idx, ]

  hm_list <- plotMotifHeatmaps(
    x           = se_sub,
    which.plots = c("log2enr", "negLog10Padj"),
    cluster     = TRUE,
    show_motif_GC = FALSE,
    show_seqlogo  = TRUE,
    column_names_rot = 45,
    column_names_gp  = grid::gpar(fontsize = 10)
  )

  ht_list <- Reduce(ComplexHeatmap::add_heatmap, hm_list)

  dir.create(dirname(filename), showWarnings = FALSE, recursive = TRUE)
  pdf(filename, width = 16, height = 12)
  draw(ht_list, padding = unit(c(2, 2, 40, 2), "mm"))
  dev.off()
  message("  Native monaLisa saved: ", basename(filename))
}

# Plot for cell types and modules
plot_monalisa_native(se_ct,
    file.path(OUT_DIR, "plots", "Native_monaLisa_CellTypes.pdf"), top_n = 35)
plot_monalisa_native(se_mod,
    file.path(OUT_DIR, "plots", "Native_monaLisa_Modules.pdf"), top_n = 35)

message("Native monaLisa logo heatmaps complete.")

ERROR: Error in h(simpleError(msg, call)): error in evaluating the argument 'x' in selecting a method for function 'assay': object 'se_ct' not found


In [ ]:
# =============================================================================
# Cell 12: Final Summary
# =============================================================================
message("\n=== Notebook 10c complete ===")
message("Modules extracted: ", length(keep_modules))
message("Total regions in modules: ", nrow(master_df))
message("\nOutput directories:")
message("  Modules:    ", file.path(OUT_DIR, "differential_results/cross_celltype_modules/"))
message("  Enrichment: ", file.path(OUT_DIR, "differential_results/enrichment/"))
message("  Motifs:     ", file.path(OUT_DIR, "differential_results/motifs/"))
message("  Plots:      ", file.path(OUT_DIR, "plots/"))